# 11 · External validity: do scores track independent signs of condition?

If inspector effects are noise on top of real condition, removing them should make a
property's score agree better with independent measures. Two versions of each property's
score:

- **raw:** mean graded score across its inspections;
- **inspector-adjusted:** the property effect from a property + inspector + year fit
  (`reliability.two_way_effects`), which removes who inspected and when.

Independent measures:

- HUD REAC physical inspection scores (National Housing Preservation Database, Dallas
  extract of 2026-09-09; standardized federal protocol; subsidized properties only);
- DCAD appraiser condition, quality and physical depreciation;
- CoStar star rating and class (2026-03 snapshot);
- eviction filings per 100 units per year, 2021–2025;
- resident 311 Code Compliance requests within 100 m per 100 units per year, 2021–2025, and the
  Emergency-priority subset;
- DPD incidents at apartment premises within 150 m, 2023–2025.

Spearman correlations. Bootstrap intervals for the difference resample properties.

## Findings

Run 2026-10-05. Outputs are cleared before commit; the numbers below are from that run.

**Raw scores track independent measures of condition modestly. Removing inspector effects
makes that agreement worse, because inspector effects carry real signal: strict inspectors
work where conditions are worse, and partly find more.**

- **Raw property scores against independent measures** (Spearman, signed so that positive
  means agreement):

  | Measure | Correlation |
  |---|---|
  | DCAD quality | 0.34 |
  | DCAD condition | 0.31 |
  | Evictions per unit | 0.29 |
  | DCAD physical depreciation | 0.24 |
  | Apartment-premise crime | 0.22 |
  | HUD REAC | 0.18 (46 properties; REAC dates span 2010–2026) |
  | Emergency 311 | 0.14 |
  | Resident 311 | 0.09 |
  | CoStar stars | 0.06 |

- **Removing inspector effects lowers every one.** DCAD condition falls from 0.31 to 0.15
  and evictions from 0.29 to 0.15. Every interval excludes zero except REAC's. The pattern
  holds among the properties seen by two or more inspectors.
- **Why.** Across 31 inspectors with 50+ inspections:
  - **Assignment.** An inspector's leniency correlates −0.57 with the eviction rate of the
    properties they inspect. Strict inspectors work where distress is higher, which fits
    Code's statement that assignments shift with enforcement needs.
  - **Detection, partly.** The strictest half's scores track DCAD condition better than the
    most lenient half's (median within-inspector correlation 0.24 against 0.12). They track
    evictions slightly better (0.20 against 0.17) and crime worse (0.11 against 0.19).
- **Reading.** An inspector's effect mixes three things: leniency, where they are sent, and
  how much they find. A fixed-effects adjustment can't separate them, so it removes real
  signal along with bias. The score has real but weak validity (about 0.3 with the best
  independent measures). Its reliability across inspectors is low (notebook 08).

In [ ]:
from __future__ import annotations

import warnings

import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

import features as f
import holdout as h
import reliability as rl
import temporal as t

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.3f}".format)

insp = t.load_inspections()
insp = insp[insp.inspector >= 0]
eff = rl.two_way_effects(insp)
prop = insp.groupby("address").agg(raw=("score", "mean"), n_insp=("score", "size"),
                                   n_inspectors=("inspector", "nunique"))
prop["adjusted"] = eff["unit"].reindex(prop.index) + insp.score.mean()

base = h.load_base()
cs, _ = f.costar(base)
ext = pd.concat([
    base[["address", "latitude", "longitude", "dcad_cond", "dcad_qual", "dcad_phys_depr_pct"]],
    cs[["costar_stars", "costar_class"]],
    f.evictions(base)[["ev_during_adj"]],
    f.complaints_311(base)[["c311_during", "c311_emergency_during"]],
    f.apartment_crime(base)[["crime_apartment_2023_25"]],
], axis=1).set_index("address")

In [ ]:
# HUD REAC: latest score per NHPD property, matched to the nearest inspected property within 75 m
n = pd.read_excel(h.DATA / "nhpd_properties-dallas_tx-20260909.xlsx")
n = n.dropna(subset=["REAC Score 1", "Latitude", "Longitude"]).copy()
n["reac"] = pd.to_numeric(n["REAC Score 1"].astype(str).str.extract(r"^(\d+)")[0], errors="coerce")
n["reac_date"] = pd.to_datetime(n["Last REAC Date"], errors="coerce")
tree = BallTree(np.radians(n[["Latitude", "Longitude"]].to_numpy(float)), metric="haversine")
xy = ext[["latitude", "longitude"]].dropna()
dist, idx = tree.query(np.radians(xy.to_numpy(float)), k=1)
ok = dist[:, 0] * f.EARTH_KM * 1000 <= 75
ext["reac"] = np.nan
ext.loc[xy.index[ok], "reac"] = n.reac.to_numpy()[idx[ok, 0]]
reac_dates = n.reac_date.to_numpy()[idx[ok, 0]]
print(f"NHPD properties with a REAC score: {len(n)}; matched to an inspected property: {int(ok.sum())}; "
      f"REAC dates {pd.Series(reac_dates).min():%Y-%m} to {pd.Series(reac_dates).max():%Y-%m}")

In [ ]:
MEASURES = {
    "HUD REAC score": ("reac", +1),
    "DCAD condition": ("dcad_cond", +1),
    "DCAD quality": ("dcad_qual", +1),
    "DCAD physical depreciation": ("dcad_phys_depr_pct", -1),
    "CoStar stars": ("costar_stars", +1),
    "CoStar class": ("costar_class", +1),
    "evictions per unit, 2021-25": ("ev_during_adj", -1),
    "resident 311 per unit, 2021-25": ("c311_during", -1),
    "Emergency 311 per unit, 2021-25": ("c311_emergency_during", -1),
    "apartment-premise crime, 2023-25": ("crime_apartment_2023_25", -1),
}
d = prop.join(ext, how="inner")


def spearman(x: pd.DataFrame, a: str, b: str) -> float:
    return x[a].corr(x[b], method="spearman")


rows = []
for name, (col, sign) in MEASURES.items():
    dd = d.dropna(subset=[col]).reset_index()
    raw = sign * spearman(dd, "raw", col)
    adj = sign * spearman(dd, "adjusted", col)
    diff, lo, hi = rl.cluster_bootstrap(
        dd, lambda x: sign * (spearman(x, "adjusted", col) - spearman(x, "raw", col)), n=500)
    rows.append({"measure": name, "properties": len(dd), "raw": raw, "adjusted": adj,
                 "adjusted - raw": diff, "95% low": lo, "95% high": hi})
res = pd.DataFrame(rows)
print("Spearman correlation with each property's score, signed so that positive = agrees with better condition")
print(res.round(3).to_string(index=False))

In [ ]:
# The adjustment can only help where a property has been seen by more than one inspector.
multi = d[d.n_inspectors >= 2]
rows = []
for name, (col, sign) in MEASURES.items():
    dd = multi.dropna(subset=[col])
    if len(dd) < 30:
        continue
    rows.append({"measure": name, "properties": len(dd),
                 "raw": sign * spearman(dd, "raw", col), "adjusted": sign * spearman(dd, "adjusted", col)})
print("properties inspected by 2+ inspectors")
print(pd.DataFrame(rows).round(3).to_string(index=False))

## Is a strict inspector biased, or better at finding problems?

Removing inspector effects can lower agreement with independent measures in two ways:
strict inspectors work where conditions are worse, or they detect problems lenient
inspectors miss, so their scores carry more signal. Two checks across inspectors with 50+
inspections:

- **Where they work:** an inspector's effect against the average condition of the properties
  they inspect.
- **How well their scores track condition:** within each inspector's own inspections, the
  Spearman correlation between score and each independent measure, against their effect.

In [ ]:
iv = insp.join(ext, on="address")
n_by = iv.inspector.value_counts()
big = n_by[n_by >= 50].index
rows = []
for k in big:
    x = iv[iv.inspector == k]
    rows.append({"inspector": k, "inspections": len(x), "effect": eff["rater"].get(k),
                 "mean DCAD condition": x.dcad_cond.mean(), "mean evictions": x.ev_during_adj.mean(),
                 "share high-poverty": (x.address.map(base.set_index("address").pct_poverty) >= h.poverty_cut(base)).mean(),
                 "validity: DCAD condition": x.score.corr(x.dcad_cond, method="spearman"),
                 "validity: evictions": -x.score.corr(x.ev_during_adj, method="spearman"),
                 "validity: crime": -x.score.corr(x.crime_apartment_2023_25, method="spearman")})
per = pd.DataFrame(rows)
print(f"{len(per)} inspectors with 50+ inspections")
print("correlation across inspectors between leniency (effect) and:")
for c in ["mean DCAD condition", "mean evictions", "share high-poverty",
          "validity: DCAD condition", "validity: evictions", "validity: crime"]:
    print(f"  {c:28s} {per.effect.corr(per[c], method='spearman'):+.2f}")
print("\nwithin-inspector validity, median across inspectors:")
print(per[["validity: DCAD condition", "validity: evictions", "validity: crime"]].median().round(3).to_string())
lo_h = per.effect <= per.effect.median()
print("\nmedian validity, strictest half vs most lenient half of inspectors:")
print(pd.DataFrame({"strictest half": per[lo_h][["validity: DCAD condition", "validity: evictions", "validity: crime"]].median(),
                    "most lenient half": per[~lo_h][["validity: DCAD condition", "validity: evictions", "validity: crime"]].median()}).round(3).to_string())